In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [ ]:
# Config

@dataclass
class ToyTransformerConfig:

    d_model: int = 384
    n_heads: int = 6
    d_head: int = 64
    n_layers: int = 6
    ffn_mult: float = 4.0

    vocab_size: int = 50257
    max_seq_len: int = 1024

    dropout: float = 0.0
    tie_embeddings: bool = True
    norm_eps: float = 1e-6

    @classmethod
    def full(cls, **overrides) -> "ToyTransformerConfig":
        # Stronger baseline. Not exactly 54M because GPT-2 vocab embedding is large.
        defaults = dict(
            d_model=512,
            n_heads=8,
            d_head=64,
            n_layers=8,
            ffn_mult=4.0,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def half(cls, **overrides) -> "ToyTransformerConfig":
        # Main recommended half baseline.
        defaults = dict(
            d_model=384,
            n_heads=6,
            d_head=64,
            n_layers=6,
            ffn_mult=4.0,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def half_param_matched(cls, **overrides) -> "ToyTransformerConfig":
        # More parameter-conscious comparison against ToyMamba2/ToyKimi half runs.
        defaults = dict(
            d_model=384,
            n_heads=6,
            d_head=64,
            n_layers=4,
            ffn_mult=4.0,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def quarter(cls, **overrides) -> "ToyTransformerConfig":
        defaults = dict(
            d_model=256,
            n_heads=4,
            d_head=64,
            n_layers=4,
            ffn_mult=4.0,
        )
        defaults.update(overrides)
        return cls(**defaults)

    def validate(self) -> None:
        assert self.d_model == self.n_heads * self.d_head, (
            f"d_model must equal n_heads * d_head, got "
            f"{self.d_model} != {self.n_heads} * {self.d_head}"
        )
        assert self.max_seq_len > 0
        assert self.vocab_size > 0
        assert self.n_layers > 0
        assert self.ffn_mult > 0

    @property
    def ffn_dim(self) -> int:
        return int(self.d_model * self.ffn_mult)

In [ ]:
# Model

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(-1, keepdim=True) + self.eps)
        out = x_float * rms * self.weight.float()
        return out.to(dtype)


class RotaryEmbedding(nn.Module):
    """
    RoPE cache that can safely extend if a longer sequence is seen.
    """

    def __init__(self, dim: int, max_seq_len: int = 4096, base: float = 10000.0):
        super().__init__()
        self.dim = dim
        self.base = base
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.register_buffer("cos_cached", torch.empty(0), persistent=False)
        self.register_buffer("sin_cached", torch.empty(0), persistent=False)
        self._build_cache(max_seq_len, device=inv_freq.device, dtype=torch.float32)

    def _build_cache(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> None:
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq.to(device=device))
        emb = torch.cat([freqs, freqs], dim=-1)
        self.cos_cached = emb.cos().to(dtype=dtype)
        self.sin_cached = emb.sin().to(dtype=dtype)

    def forward(
        self,
        seq_len: int,
        device: torch.device,
        dtype: torch.dtype,
        offset: int = 0,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        needed = offset + seq_len
        if self.cos_cached.numel() == 0 or needed > self.cos_cached.size(0):
            new_len = max(needed, 2 * max(1, self.cos_cached.size(0)))
            self._build_cache(new_len, device=device, dtype=torch.float32)

        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(
    q: torch.Tensor,
    k: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    # q, k: [B, H, T, D]
    # cos, sin: [T, D]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    q = q * cos + rotate_half(q) * sin
    k = k * cos + rotate_half(k) * sin
    return q, k


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: ToyTransformerConfig):
        super().__init__()
        d, h, dh = cfg.d_model, cfg.n_heads, cfg.d_head
        self.h = h
        self.dh = dh

        self.W_q = nn.Linear(d, h * dh, bias=False)
        self.W_k = nn.Linear(d, h * dh, bias=False)
        self.W_v = nn.Linear(d, h * dh, bias=False)
        self.W_o = nn.Linear(h * dh, d, bias=False)

        self.rope = RotaryEmbedding(dh, max_seq_len=cfg.max_seq_len)
        self.dropout = cfg.dropout

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> tuple[torch.Tensor, Optional[tuple[torch.Tensor, torch.Tensor]]]:
        B, T, _ = x.shape
        h, dh = self.h, self.dh

        q = self.W_q(x).view(B, T, h, dh).transpose(1, 2)  # [B,H,T,D]
        k = self.W_k(x).view(B, T, h, dh).transpose(1, 2)
        v = self.W_v(x).view(B, T, h, dh).transpose(1, 2)

        past_len = 0 if kv_cache is None else kv_cache[0].size(2)
        cos, sin = self.rope(
            seq_len=T,
            device=x.device,
            dtype=q.dtype,
            offset=past_len,
        )
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)

        new_cache = (k, v) if use_cache else None

        # Full sequence training/prefill: causal mask.
        # Cached decoding: current token attends to all cached keys, so no causal mask needed.
        is_causal = kv_cache is None and T > 1

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )
        out = out.transpose(1, 2).reshape(B, T, h * dh)
        return self.W_o(out), new_cache


class SwiGLUFFN(nn.Module):
    def __init__(self, cfg: ToyTransformerConfig):
        super().__init__()
        d, ffn = cfg.d_model, cfg.ffn_dim
        self.W_gate = nn.Linear(d, ffn, bias=False)
        self.W_up = nn.Linear(d, ffn, bias=False)
        self.W_down = nn.Linear(ffn, d, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.silu(self.W_gate(x)) * self.W_up(x)
        x = self.W_down(x)
        return self.dropout(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: ToyTransformerConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = CausalSelfAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLUFFN(cfg)

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> tuple[torch.Tensor, Optional[tuple[torch.Tensor, torch.Tensor]]]:
        attn_out, new_cache = self.attn(
            self.attn_norm(x),
            kv_cache=kv_cache,
            use_cache=use_cache,
        )
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_cache


class ToyRoPETransformer(nn.Module):
    def __init__(self, cfg: ToyTransformerConfig):
        super().__init__()
        cfg.validate()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.apply(self._init_weights)

        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

    def _init_weights(self, m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(
        self,
        input_ids: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
        kv_caches: Optional[list[Optional[tuple[torch.Tensor, torch.Tensor]]]] = None,
        use_cache: bool = False,
    ) -> tuple[
        torch.Tensor,
        Optional[torch.Tensor],
        Optional[list[Optional[tuple[torch.Tensor, torch.Tensor]]]],
    ]:
        x = self.tok_emb(input_ids)

        new_caches = [] if use_cache else None
        if kv_caches is None:
            kv_caches = [None] * len(self.layers)

        for layer, cache in zip(self.layers, kv_caches):
            x, new_cache = layer(x, kv_cache=cache, use_cache=use_cache)
            if use_cache:
                new_caches.append(new_cache)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )

        return logits, loss, new_caches

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 128,
        temperature: float = 0.8,
        top_k: int = 40,
        use_cache: bool = True,
    ) -> torch.Tensor:
        self.eval()

        if not use_cache:
            return self._generate_naive(
                input_ids=input_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_k=top_k,
            )

        idx = input_ids[:, -self.cfg.max_seq_len:]
        logits, _, caches = self.forward(idx, use_cache=True)
        next_logits = logits[:, -1, :] / max(temperature, 1e-8)

        generated = input_ids
        for _ in range(max_new_tokens):
            if top_k > 0:
                values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits = next_logits.masked_fill(next_logits < values[:, [-1]], -float("inf"))

            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_id], dim=1)

            # If cache grows past max_seq_len, recompute recent context.
            # This keeps generation safe without implementing sliding KV eviction.
            if generated.size(1) > self.cfg.max_seq_len:
                idx = generated[:, -self.cfg.max_seq_len:]
                logits, _, caches = self.forward(idx, use_cache=True)
            else:
                logits, _, caches = self.forward(next_id, kv_caches=caches, use_cache=True)

            next_logits = logits[:, -1, :] / max(temperature, 1e-8)

        return generated

    @torch.no_grad()
    def _generate_naive(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int,
        temperature: float,
        top_k: int,
    ) -> torch.Tensor:
        self.eval()
        for _ in range(max_new_tokens):
            idx = input_ids[:, -self.cfg.max_seq_len:]
            logits, _, _ = self.forward(idx)
            logits = logits[:, -1, :] / max(temperature, 1e-8)

            if top_k > 0:
                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits = logits.masked_fill(logits < values[:, [-1]], -float("inf"))

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_id], dim=1)

        return input_ids

    def num_parameters(self, only_trainable: bool = True) -> int:
        return sum(
            p.numel()
            for p in self.parameters()
            if (not only_trainable or p.requires_grad)
        )

In [ ]:
# Data Preparation and handling

class PackedTokenDataset(Dataset):
   # Concats token ids and slices them into fixed-length windows

    def __init__(self, token_ids: list[int], seq_len: int):
        if len(token_ids) < seq_len + 1:
            raise ValueError(
                f"Need at least seq_len + 1 tokens, got {len(token_ids)} tokens "
                f"for seq_len={seq_len}."
            )
        self.data = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len
        self.n_samples = (len(self.data) - 1) // seq_len

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        start = idx * self.seq_len
        chunk = self.data[start : start + self.seq_len + 1]
        return chunk[:-1], chunk[1:]


def _row_to_text(row: dict) -> str:
    for key in ("text", "content", "story"):
        value = row.get(key)
        if isinstance(value, str):
            return value
    return ""


def load_tokens(dataset_name: str, tokenizer, split: str = "train", max_tokens: Optional[int] = None,) -> list[int]:
    # Returns a flat list of token ids for a file or HF dataset split

    from datasets import load_dataset

    if dataset_name.startswith("file:"):
        path = dataset_name[5:]
        with open(path, encoding="utf-8") as f:
            text = f.read()
        tokens = tokenizer.encode(text)
        return tokens[:max_tokens] if max_tokens else tokens

    ds_configs = {
        "wikitext-2": ("wikitext", "wikitext-2-raw-v1", False),
        "wikitext-103": ("wikitext", "wikitext-103-raw-v1", False),
        "tinystories": ("roneneldan/TinyStories", None, False),
        "openwebtext": ("Skylion007/openwebtext", None, True),
        "fineweb-edu": ("HuggingFaceFW/fineweb-edu-score-2", "sample-10BT", True),
        "slimpajama": ("cerebras/SlimPajama-627B", None, True),
        "c4": ("allenai/c4", "en", True),
    }

    if dataset_name not in ds_configs:
        raise ValueError(
            f"Unknown dataset '{dataset_name}'. Available: {', '.join(ds_configs)}, "
            "or file:<path>."
        )

    ds_name, ds_config, streaming = ds_configs[dataset_name]
    tokens: list[int] = []
    limit = max_tokens or 500_000_000

    if streaming:
        print(f"Loading {dataset_name} in streaming mode ...")
        ds = load_dataset(ds_name, ds_config, split=split, streaming=True)
        for row in ds:
            text = _row_to_text(row)
            if text.strip():
                tokens.extend(tokenizer.encode(text))
                if len(tokens) >= limit:
                    tokens = tokens[:limit]
                    break
    else:
        ds = load_dataset(ds_name, ds_config, split=split)
        print(f"Tokenising {dataset_name} / {split} ({len(ds)} rows) ...")
        for row in ds:
            text = _row_to_text(row)
            if text.strip():
                tokens.extend(tokenizer.encode(text))
                if len(tokens) >= limit:
                    tokens = tokens[:limit]
                    break

    print(f"  -> {len(tokens):,} tokens")
    return tokens


def make_data_splits(
    dataset_name: str,
    tokenizer,
    max_tokens: Optional[int],
    val_monitor_fraction: float,
    val_ablation_fraction: float,
    test_replication_fraction: float,
    val_monitor_max_tokens: Optional[int],
    val_ablation_max_tokens: Optional[int],
    test_replication_max_tokens: Optional[int],
) -> tuple[dict[str, list[int]], dict]:
    """
    Creates train / val_monitor / val_ablation / test_replication token splits.

    Expected usage:
      - train: backprop only
      - val_monitor: early stopping and best-checkpoint selection
      - val_ablation: compare architecture/hyperparameter variants
      - test_replication: final report only

    Split policy:
      - For Wikitext, use official validation for val_monitor and official test
        for test_replication. val_ablation is carved from the tail of train.
      - For file:<path>, TinyStories, and most other datasets, load the train
        split and reserve the tail for held-out sets.
      - For serious TinyStories experiments, a story-level split is cleaner than
        this token-level split, but this version is simple.
    """
    split_info = {
        "dataset": dataset_name,
        "split_policy": None,
        "fractions": {
            "val_monitor": val_monitor_fraction,
            "val_ablation": val_ablation_fraction,
            "test_replication": test_replication_fraction,
        },
    }

    def cap_tokens(tokens: list[int], cap: Optional[int]) -> list[int]:
        return tokens[:cap] if cap is not None else tokens

    if dataset_name in {"wikitext-2", "wikitext-103"}:
        train_all = load_tokens(dataset_name, tokenizer, split="train", max_tokens=max_tokens)
        val_monitor = load_tokens(
            dataset_name,
            tokenizer,
            split="validation",
            max_tokens=val_monitor_max_tokens,
        )
        test_replication = load_tokens(
            dataset_name,
            tokenizer,
            split="test",
            max_tokens=test_replication_max_tokens,
        )

        n_ablation = max(int(len(train_all) * val_ablation_fraction), 1)
        if val_ablation_max_tokens is not None:
            n_ablation = min(n_ablation, val_ablation_max_tokens)

        train = train_all[:-n_ablation]
        val_ablation = train_all[-n_ablation:]

        split_info["split_policy"] = "official validation/test; val_ablation carved from train tail"
    else:
        all_tokens = load_tokens(dataset_name, tokenizer, split="train", max_tokens=max_tokens)
        n_total = len(all_tokens)

        n_monitor = max(int(n_total * val_monitor_fraction), 1)
        n_ablation = max(int(n_total * val_ablation_fraction), 1)
        n_test = max(int(n_total * test_replication_fraction), 1)

        if val_monitor_max_tokens is not None:
            n_monitor = min(n_monitor, val_monitor_max_tokens)
        if val_ablation_max_tokens is not None:
            n_ablation = min(n_ablation, val_ablation_max_tokens)
        if test_replication_max_tokens is not None:
            n_test = min(n_test, test_replication_max_tokens)

        heldout_total = n_monitor + n_ablation + n_test
        if n_total - heldout_total < 2:
            raise ValueError(
                "Training split is too small after held-out splits. "
                "Increase max_tokens or reduce split fractions."
            )

        train_end = n_total - heldout_total
        monitor_end = train_end + n_monitor
        ablation_end = monitor_end + n_ablation

        train = all_tokens[:train_end]
        val_monitor = all_tokens[train_end:monitor_end]
        val_ablation = all_tokens[monitor_end:ablation_end]
        test_replication = all_tokens[ablation_end:]

        split_info["split_policy"] = "tail holdout from train token stream"
        split_info["n_total_loaded_tokens"] = n_total

    splits = {
        "train": train,
        "val_monitor": cap_tokens(val_monitor, val_monitor_max_tokens),
        "val_ablation": cap_tokens(val_ablation, val_ablation_max_tokens),
        "test_replication": cap_tokens(test_replication, test_replication_max_tokens),
    }

    split_info["n_tokens"] = {name: len(tokens) for name, tokens in splits.items()}
    split_info["heldout_names"] = ["val_monitor", "val_ablation", "test_replication"]
    return splits, split_info


def verify_splits(splits: dict[str, list[int]],seq_len: int,) -> dict:
    # Sanity-checks all data splits. Because every split is turned into its own PackedTokenDataset, no packed
    # training window can cross from one split into another.

    required = ["train", "val_monitor", "val_ablation", "test_replication"]
    for name in required:
        if name not in splits:
            raise KeyError(f"Missing split: {name}")
        if len(splits[name]) < seq_len + 1:
            raise ValueError(
                f"Split '{name}' too small: {len(splits[name])} tokens for seq_len={seq_len}."
            )

    checks = {}
    for name, tokens in splits.items():
        checks[name] = {
            "tokens": len(tokens),
            "samples": (len(tokens) - 1) // seq_len,
        }

    checks["windows_cross_boundary"] = False
    checks["note"] = (
        "PackedTokenDataset is built separately for each split, so windows do not cross boundaries. "
        "This does not prove semantic deduplication."
    )
    return checks

In [ ]:
# Utilities for training

def cosine_with_warmup(step: int, warmup: int, total: int) -> float:
    if step < warmup:
        return step / max(warmup, 1)
    progress = (step - warmup) / max(total - warmup, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def gpu_stats(device: torch.device) -> dict:
    if device.type != "cuda":
        return {}
    return {
        "gpu_mem_allocated_gb": round(torch.cuda.memory_allocated(device) / 1e9, 3),
        "gpu_mem_reserved_gb": round(torch.cuda.memory_reserved(device) / 1e9, 3),
        "gpu_mem_peak_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 3),
    }


def estimate_transformer_flops_per_step(cfg: ToyTransformerConfig, effective_batch_size: int) -> dict:
    """
    Approximate training FLOPs.

    This is for comparison logging, not an exact profiler.
    """
    T = cfg.max_seq_len
    d = cfg.d_model
    h = cfg.n_heads
    dh = cfg.d_head
    L = cfg.n_layers
    ffn = cfg.ffn_dim

    # Attention matmuls: QK^T and AV.
    flops_attn = L * h * 2 * T * T * dh

    # Q,K,V,O projections.
    flops_proj = L * 4 * 2 * T * d * (h * dh)

    # SwiGLU: gate, up, down.
    flops_ffn = L * 3 * 2 * T * d * ffn

    # Embedding lookup is not really matmul FLOPs. LM head is counted.
    flops_lm_head = 2 * T * d * cfg.vocab_size

    flops_fwd = flops_attn + flops_proj + flops_ffn + flops_lm_head
    flops_step = 3 * flops_fwd * effective_batch_size

    return {
        "flops_fwd_per_sample": flops_fwd,
        "flops_per_step": flops_step,
        "breakdown": {
            "attention": flops_attn,
            "projections": flops_proj,
            "ffn": flops_ffn,
            "lm_head": flops_lm_head,
        },
    }


class TrainingLogger:
    def __init__(self, log_path: str, run_config: dict):
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self.f = open(self.log_path, "w", encoding="utf-8")
        self._write({
            "type": "run_config",
            "timestamp": datetime.now().isoformat(),
            **run_config,
        })

    def log_step(self, metrics: dict) -> None:
        self._write({"type": "step", "timestamp": datetime.now().isoformat(), **metrics})

    def log_eval(self, metrics: dict) -> None:
        self._write({"type": "eval", "timestamp": datetime.now().isoformat(), **metrics})

    def log_final(self, metrics: dict) -> None:
        self._write({"type": "final", "timestamp": datetime.now().isoformat(), **metrics})

    def _write(self, d: dict) -> None:
        self.f.write(json.dumps(d, default=str) + "\n")
        self.f.flush()

    def close(self) -> None:
        self.f.close()


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    amp_dtype: torch.dtype,
    max_batches: Optional[int] = None,
    use_amp: bool = True,
) -> dict:
    model.eval()
    losses = []
    n_tokens = 0
    t0 = time.time()

    for i, (x, y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.amp.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
            _, loss, _ = model(x, y)

        losses.append(float(loss.item()))
        n_tokens += x.numel()

    if not losses:
        raise RuntimeError("Evaluation loader produced no batches.")

    val_loss = sum(losses) / len(losses)
    return {
        "val_loss": round(val_loss, 5),
        "val_ppl": round(math.exp(min(val_loss, 20)), 3),
        "val_batches": len(losses),
        "val_tokens": n_tokens,
        "val_time_sec": round(time.time() - t0, 2),
    }


def save_checkpoint(
    path: str | Path,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer],
    scheduler: Optional[torch.optim.lr_scheduler.LRScheduler],
    cfg: ToyTransformerConfig,
    step: int,
    extra: Optional[dict] = None,
) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "step": step,
        "optimizer_step": step,
        "model": model.state_dict(),
        "config": asdict(cfg),
    }
    if optimizer is not None:
        payload["optimizer"] = optimizer.state_dict()
    if scheduler is not None:
        payload["scheduler"] = scheduler.state_dict()
    if extra:
        payload["extra"] = extra

    torch.save(payload, path)
    print(f"saved checkpoint: {path}")

In [ ]:
# Config

@dataclass
class ColabRunConfig:
    MODEL_SIZE: str = "half_param_matched"  # "quarter", "half_param_matched", "half", "full"

    DATASET: str = "tinystories"
    MAX_TOKENS: Optional[int] = None

    # Held-out split fractions. These mirror ToyMamba2 and ToyKimiLinear.
    VAL_MONITOR_FRACTION: float = 0.02
    VAL_ABLATION_FRACTION: float = 0.015
    TEST_REPLICATION_FRACTION: float = 0.015

    VAL_MONITOR_MAX_TOKENS: Optional[int] = 2_000_000
    VAL_ABLATION_MAX_TOKENS: Optional[int] = 1_000_000
    TEST_REPLICATION_MAX_TOKENS: Optional[int] = 1_000_000

    SEQ_LEN: int = 1024
    BATCH_SIZE: int = 24
    EVAL_BATCH_SIZE: Optional[int] = None
    GRAD_ACCUM: int = 1
    NUM_WORKERS: int = 2

    LR: float = 3e-4
    WEIGHT_DECAY: float = 0.01
    WARMUP_STEPS: int = 1000
    MAX_STEPS: int = 100_000
    GRAD_CLIP: float = 1.0

    LOG_EVERY: int = 100
    EVAL_EVERY: int = 500
    EVAL_BATCHES: int = 50
    FINAL_EVAL_BATCHES: int = 200
    SAVE_EVERY: int = 5000

    MIN_EVALS_BEFORE_STOP: int = 6
    EARLY_STOP_PATIENCE: int = 8
    EARLY_STOP_MIN_DELTA: float = 0.002

    SAVE_ROOT: str = "/content/drive/MyDrive/models/transformer_toy2/checkpoints"
    LOG_ROOT: str = "/content/drive/MyDrive/models/transformer_toy2/logs"

    RUN_CORRECTNESS_TESTS: bool = True


def make_colab_config() -> ColabRunConfig:
    return ColabRunConfig()

In [ ]:
# Correctness tests

@torch.no_grad()
def run_correctness_tests(device: torch.device) -> None:
    print("Running correctness tests...")

    cfg = ToyTransformerConfig.quarter(
        max_seq_len=64,
        vocab_size=256,
        n_layers=2,
        dropout=0.0,
    )
    model = ToyRoPETransformer(cfg).to(device)
    model.eval()

    x = torch.randint(0, cfg.vocab_size, (2, 16), device=device)

    # Full forward.
    logits_full, _, _ = model(x)

    # Cached prefill should match full forward.
    logits_cache, _, _ = model(x, use_cache=True)
    max_diff_prefill = (logits_full - logits_cache).abs().max().item()

    # Step-by-step cached path should match full forward reasonably.
    caches = None
    logits_steps = []
    for t in range(x.size(1)):
        logits_t, _, caches = model(x[:, t:t + 1], kv_caches=caches, use_cache=True)
        logits_steps.append(logits_t)
    logits_steps = torch.cat(logits_steps, dim=1)
    max_diff_steps = (logits_full - logits_steps).abs().max().item()

    print(f"  prefill max diff: {max_diff_prefill:.6f}")
    print(f"  step cache max diff: {max_diff_steps:.6f}")

    # In bf16/cuda, tiny differences are normal because attention kernels differ.
    assert max_diff_prefill < 1e-3, "Prefill cache does not match full forward."
    assert max_diff_steps < 1e-2, "Step cache does not match full forward."

    print("Correctness tests passed.")

In [ ]:
# Training

def train(run: Optional[ColabRunConfig] = None) -> None:
    run = run or make_colab_config()

    if torch.cuda.is_available():
        device = torch.device("cuda")
        amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        use_amp = True
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
        # Safer default; AMP on MPS can be version-dependent.
        amp_dtype = torch.float32
        use_amp = False
    else:
        device = torch.device("cpu")
        amp_dtype = torch.float32
        use_amp = False

    print(f"Device: {device} | AMP dtype: {amp_dtype} | use_amp: {use_amp}")

    factory = {
        "quarter": ToyTransformerConfig.quarter,
        "half_param_matched": ToyTransformerConfig.half_param_matched,
        "half": ToyTransformerConfig.half,
        "full": ToyTransformerConfig.full,
    }[run.MODEL_SIZE]

    cfg = factory(max_seq_len=run.SEQ_LEN)
    cfg.validate()

    if run.RUN_CORRECTNESS_TESTS:
        run_correctness_tests(device)

    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("gpt2", model_max_length=1_000_000)

    token_splits, split_info = make_data_splits(
        dataset_name=run.DATASET,
        tokenizer=tokenizer,
        max_tokens=run.MAX_TOKENS,
        val_monitor_fraction=run.VAL_MONITOR_FRACTION,
        val_ablation_fraction=run.VAL_ABLATION_FRACTION,
        test_replication_fraction=run.TEST_REPLICATION_FRACTION,
        val_monitor_max_tokens=run.VAL_MONITOR_MAX_TOKENS,
        val_ablation_max_tokens=run.VAL_ABLATION_MAX_TOKENS,
        test_replication_max_tokens=run.TEST_REPLICATION_MAX_TOKENS,
    )
    split_checks = verify_splits(token_splits, cfg.max_seq_len)

    datasets = {
        name: PackedTokenDataset(tokens, cfg.max_seq_len)
        for name, tokens in token_splits.items()
    }

    print("Dataset windows:", {name: len(ds) for name, ds in datasets.items()})
    print("Split info:", split_info)
    print("Split checks:", split_checks)

    train_loader = DataLoader(
        datasets["train"],
        batch_size=run.BATCH_SIZE,
        shuffle=True,
        num_workers=run.NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        drop_last=True,
    )

    def eval_loader(name: str) -> DataLoader:
        return DataLoader(
            datasets[name],
            batch_size=run.EVAL_BATCH_SIZE or run.BATCH_SIZE,
            shuffle=False,
            num_workers=run.NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
            drop_last=False,
        )

    val_monitor_loader = eval_loader("val_monitor")
    val_ablation_loader = eval_loader("val_ablation")
    test_replication_loader = eval_loader("test_replication")

    model = ToyRoPETransformer(cfg).to(device)
    n_params = model.num_parameters()

    print(f"Model parameters: {n_params:,} ({n_params / 1e6:.2f}M)")

    effective_batch = run.BATCH_SIZE * max(1, run.GRAD_ACCUM)
    tokens_per_step = effective_batch * cfg.max_seq_len
    total_training_tokens = run.MAX_STEPS * tokens_per_step
    flops_info = estimate_transformer_flops_per_step(cfg, effective_batch)

    print(f"Tokens per step: {tokens_per_step:,}")
    print(f"Max planned tokens: {total_training_tokens:,} ({total_training_tokens / 1e9:.2f}B)")
    print(f"Chinchilla ratio: {total_training_tokens / n_params:.1f} tokens/param")
    print(f"FLOPs per step estimate: {flops_info['flops_per_step']:.2e}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=run.LR,
        betas=(0.9, 0.95),
        weight_decay=run.WEIGHT_DECAY,
        fused=(device.type == "cuda"),
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda s: cosine_with_warmup(s, run.WARMUP_STEPS, run.MAX_STEPS),
    )

    use_scaler = device.type == "cuda" and amp_dtype == torch.float16
    scaler = torch.amp.GradScaler(device.type, enabled=use_scaler)

    run_name = (
        f"toy_rope_transformer_{run.MODEL_SIZE}_{run.DATASET}_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    )
    save_dir = os.path.join(run.SAVE_ROOT, run_name)
    log_path = os.path.join(run.LOG_ROOT, f"{run_name}.jsonl")

    run_config = {
        "arch_name": "toy_rope_transformer",
        "model_size_preset": run.MODEL_SIZE,
        "model_config": asdict(cfg),
        "n_params": n_params,

        "dataset": run.DATASET,
        "split_info": split_info,
        "split_checks": split_checks,

        "batch_size": run.BATCH_SIZE,
        "eval_batch_size": run.EVAL_BATCH_SIZE,
        "grad_accum": run.GRAD_ACCUM,
        "effective_batch": effective_batch,
        "tokens_per_optimizer_step": tokens_per_step,

        "max_steps": run.MAX_STEPS,
        "lr": run.LR,
        "weight_decay": run.WEIGHT_DECAY,
        "warmup_steps": run.WARMUP_STEPS,
        "grad_clip": run.GRAD_CLIP,

        "estimated_flops": flops_info,

        "device": str(device),
        "amp_dtype": str(amp_dtype),
        "use_amp": use_amp,

        "early_stopping_patience": run.EARLY_STOP_PATIENCE,
        "early_stopping_min_delta": run.EARLY_STOP_MIN_DELTA,
        "min_evals_before_stop": run.MIN_EVALS_BEFORE_STOP,

        "log_every": run.LOG_EVERY,
        "eval_every": run.EVAL_EVERY,
        "eval_batches": run.EVAL_BATCHES,
        "final_eval_batches": run.FINAL_EVAL_BATCHES,
    }

    if device.type == "cuda":
        run_config["gpu_name"] = torch.cuda.get_device_name(device)
        run_config["gpu_memory_gb"] = round(
            torch.cuda.get_device_properties(device).total_memory / 1e9,
            1,
        )

    logger = TrainingLogger(log_path, run_config)
    print(f"Logging to: {log_path}")
    print(f"Saving to: {save_dir}")

    best_checkpoint_loss = float("inf")
    best_checkpoint_step = 0

    best_stopping_loss = float("inf")
    best_stopping_step = 0
    stale_evals = 0
    eval_count = 0
    stopped_by_early_stopping = False

    micro_steps = 0
    optimizer_steps = 0
    running_loss = 0.0
    running_steps = 0
    cumulative_flops = 0.0
    t_start = time.time()

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    print("=" * 72)
    print(f"Training ToyRoPETransformer {run.MODEL_SIZE}")
    print("=" * 72)

    # Initial validation, matching the Mamba/Kimi-style monitor split.
    initial_metrics = evaluate(
        model=model,
        loader=val_monitor_loader,
        device=device,
        amp_dtype=amp_dtype,
        max_batches=run.EVAL_BATCHES,
        use_amp=use_amp,
    )
    eval_count += 1

    best_checkpoint_loss = initial_metrics["val_loss"]
    best_checkpoint_step = 0
    best_stopping_loss = initial_metrics["val_loss"]
    best_stopping_step = 0

    logger.log_eval({
        "step": 0,
        "optimizer_step": 0,
        "split": "val_monitor",
        **initial_metrics,
        "best_checkpoint_loss": best_checkpoint_loss,
        "best_checkpoint_step": best_checkpoint_step,
        "best_stopping_loss": best_stopping_loss,
        "best_stopping_step": best_stopping_step,
        "stale_evals": stale_evals,
        "status": "too_early",
    })

    save_checkpoint(
        os.path.join(save_dir, "best.pt"),
        model,
        optimizer,
        scheduler,
        cfg,
        0,
        extra={"val_monitor": initial_metrics},
    )

    model.train()

    while optimizer_steps < run.MAX_STEPS:
        for x, y in train_loader:
            if optimizer_steps >= run.MAX_STEPS:
                break

            step_t0 = time.time()

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
                _, loss, _ = model(x, y)
                loss = loss / max(1, run.GRAD_ACCUM)

            scaler.scale(loss).backward()

            micro_steps += 1
            step_loss = float(loss.item()) * max(1, run.GRAD_ACCUM)
            running_loss += step_loss
            running_steps += 1

            grad_norm = None
            if micro_steps % run.GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    run.GRAD_CLIP,
                ).item()

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

                optimizer_steps += 1
                cumulative_flops += flops_info["flops_per_step"]

                if optimizer_steps % run.LOG_EVERY == 0:
                    avg_loss = running_loss / max(1, running_steps)
                    ppl = math.exp(min(avg_loss, 20))
                    elapsed = time.time() - t_start
                    tokens_seen = optimizer_steps * tokens_per_step
                    tok_per_sec = tokens_seen / max(elapsed, 1e-9)
                    lr_now = scheduler.get_last_lr()[0]

                    metrics = {
                        "step": optimizer_steps,
                        "optimizer_step": optimizer_steps,
                        "micro_steps": micro_steps,
                        "train_loss": round(avg_loss, 5),
                        "train_ppl": round(ppl, 3),
                        "lr": lr_now,
                        "grad_norm": round(grad_norm, 4) if grad_norm is not None else None,
                        "tokens_seen": tokens_seen,
                        "tokens_per_sec": round(tok_per_sec),
                        "step_time_ms": round((time.time() - step_t0) * 1000, 1),
                        "elapsed_min": round(elapsed / 60, 2),
                        "cumulative_flops": cumulative_flops,
                        "measured_tflops": round(cumulative_flops / max(elapsed, 1e-9) / 1e12, 2),
                        "dataset_epochs_by_tokens": round(
                            tokens_seen / max(1, len(token_splits["train"])),
                            4,
                        ),
                        **gpu_stats(device),
                    }
                    logger.log_step(metrics)

                    print(
                        f"step {optimizer_steps:>6d}/{run.MAX_STEPS} | "
                        f"train_loss {avg_loss:.4f} | train_ppl {ppl:.2f} | "
                        f"lr {lr_now:.2e} | tok/s {tok_per_sec:,.0f} | "
                        f"grad {grad_norm:.3f}"
                    )

                    running_loss = 0.0
                    running_steps = 0

                if optimizer_steps % run.EVAL_EVERY == 0:
                    val_metrics = evaluate(
                        model=model,
                        loader=val_monitor_loader,
                        device=device,
                        amp_dtype=amp_dtype,
                        max_batches=run.EVAL_BATCHES,
                        use_amp=use_amp,
                    )
                    eval_count += 1
                    val_loss = val_metrics["val_loss"]

                    improved_checkpoint = val_loss < best_checkpoint_loss
                    if improved_checkpoint:
                        best_checkpoint_loss = val_loss
                        best_checkpoint_step = optimizer_steps
                        save_checkpoint(
                            os.path.join(save_dir, "best.pt"),
                            model,
                            optimizer,
                            scheduler,
                            cfg,
                            optimizer_steps,
                            extra={"val_monitor": val_metrics},
                        )

                    meaningful_improvement = val_loss < (best_stopping_loss - run.EARLY_STOP_MIN_DELTA)
                    if meaningful_improvement:
                        best_stopping_loss = val_loss
                        best_stopping_step = optimizer_steps
                        stale_evals = 0
                    else:
                        stale_evals += 1

                    if eval_count < run.MIN_EVALS_BEFORE_STOP:
                        status = "too_early"
                    elif stale_evals >= run.EARLY_STOP_PATIENCE:
                        status = "early_stop"
                    elif meaningful_improvement:
                        status = "still_learning"
                    else:
                        status = "plateaued"

                    logger.log_eval({
                        "step": optimizer_steps,
                        "optimizer_step": optimizer_steps,
                        "split": "val_monitor",
                        **val_metrics,
                        "best_checkpoint_loss": best_checkpoint_loss,
                        "best_checkpoint_step": best_checkpoint_step,
                        "best_stopping_loss": best_stopping_loss,
                        "best_stopping_step": best_stopping_step,
                        "stale_evals": stale_evals,
                        "status": status,
                    })

                    print(
                        f"EVAL step {optimizer_steps:>6d} | "
                        f"val_loss {val_metrics['val_loss']:.5f} | "
                        f"val_ppl {val_metrics['val_ppl']:.2f} | "
                        f"best_ckpt {best_checkpoint_loss:.5f} @ step {best_checkpoint_step} | "
                        f"status {status}"
                    )

                    model.train()

                    if status == "early_stop":
                        stopped_by_early_stopping = True
                        print(
                            f"Early stopping triggered: validation loss did not improve "
                            f"by at least {run.EARLY_STOP_MIN_DELTA} for "
                            f"{run.EARLY_STOP_PATIENCE} evals."
                        )
                        break

                if run.SAVE_EVERY > 0 and optimizer_steps % run.SAVE_EVERY == 0:
                    save_checkpoint(
                        os.path.join(save_dir, f"step_{optimizer_steps}.pt"),
                        model,
                        optimizer,
                        scheduler,
                        cfg,
                        optimizer_steps,
                    )

        if stopped_by_early_stopping:
            break

    save_checkpoint(
        os.path.join(save_dir, "last.pt"),
        model,
        optimizer,
        scheduler,
        cfg,
        optimizer_steps,
    )

    # Load the numerically best validation-monitor checkpoint for final held-out evaluation.
    best_path = os.path.join(save_dir, "best.pt")
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    val_ablation_metrics = evaluate(
        model=model,
        loader=val_ablation_loader,
        device=device,
        amp_dtype=amp_dtype,
        max_batches=run.FINAL_EVAL_BATCHES,
        use_amp=use_amp,
    )

    test_replication_metrics = evaluate(
        model=model,
        loader=test_replication_loader,
        device=device,
        amp_dtype=amp_dtype,
        max_batches=run.FINAL_EVAL_BATCHES,
        use_amp=use_amp,
    )

    total_time = time.time() - t_start
    tokens_seen = optimizer_steps * tokens_per_step

    final_metrics = {
        "optimizer_steps": optimizer_steps,
        "micro_steps": micro_steps,
        "total_time_sec": round(total_time, 1),
        "total_time_hours": round(total_time / 3600, 3),
        "tokens_seen": tokens_seen,
        "avg_tokens_per_sec": round(tokens_seen / max(total_time, 1e-9)),
        "best_checkpoint_loss": best_checkpoint_loss,
        "best_checkpoint_step": best_checkpoint_step,
        "best_stopping_loss": best_stopping_loss,
        "best_stopping_step": best_stopping_step,
        "stopped_by_early_stopping": stopped_by_early_stopping,
        "val_ablation_metrics": val_ablation_metrics,
        "test_replication_metrics": test_replication_metrics,
        **gpu_stats(device),
    }

    logger.log_final(final_metrics)
    logger.close()

    print("\nTraining complete")
    print(json.dumps(final_metrics, indent=2))

    print("\nSample generation")
    prompt = "Once upon a time"
    prompt_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    out_ids = model.generate(
        prompt_ids,
        max_new_tokens=100,
        temperature=0.8,
        top_k=40,
        use_cache=True,
    )
    print(tokenizer.decode(out_ids[0].tolist()))
    print("done")

In [ ]:
train(make_colab_config())

Device: cuda | AMP dtype: torch.bfloat16 | use_amp: True
Running correctness tests...
  prefill max diff: 0.000000
  step cache max diff: 0.000001
Correctness tests passed.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenising tinystories / train (2119719 rows) ...
  -> 471,872,517 tokens
Dataset windows: {'train': 456906, 'val_monitor': 1953, 'val_ablation': 976, 'test_replication': 976}
Split info: {'dataset': 'tinystories', 'split_policy': 'tail holdout from train token stream', 'fractions': {'val_monitor': 0.02, 'val_ablation': 0.015, 'test_replication': 0.015}, 'n_total_loaded_tokens': 471872517, 'n_tokens': {'train': 467872517, 'val_monitor': 2000000, 'val_ablation': 1000000, 'test_replication': 1000000}, 'heldout_names': ['val_monitor', 'val_ablation', 'test_replication']}
Split checks: {'train': {'tokens': 467872517, 'samples': 456906}, 'val_monitor': {'tokens': 2000000, 'samples': 1953}, 'val_ablation': {'tokens': 1000000, 'samples': 976}, 'test_replication': {'tokens': 1000000, 'samples': 976}, 'windows_cross_boundary': False, 'note': 'PackedTokenDataset is built separately for each split, so windows do not cross boundaries. This does not prove semantic deduplication.'}
Model parameters: